# OLX Ko'chmas mulk — Malumot Tozalash jaroyoni  
Kodni tepadan paga qarab yurg'azing. Oxirida sizda tayor `cleaned.csv` fileli modeling uchun tayor bo'ladi.

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

In [5]:
df = pd.read_csv('../data/Praperad/olx_apartments.csv')
print(f'Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns')

Loaded: 67,219 rows, 29 columns


In [6]:
df.isna().sum()

listing_id             0
source                 0
seller_type            1
housing_type       22660
region                 1
district               1
rooms                  1
living_area_m2     45519
kitchen_area_m2    51970
total_area_m2         10
floor                  3
total_floors           1
building_type      16861
layout             23982
build_year         47336
ceiling_height     43413
bathroom           18956
furnished              5
renovation         12644
commission             1
amenities          25118
nearby             22210
negotiable             0
price                178
currency               0
published_date         1
description            1
date_scraped           0
url                    0
dtype: int64

In [7]:
df.head()

,listing_id,source,seller_type,housing_type,region,district,rooms,living_area_m2,kitchen_area_m2,total_area_m2,...,commission,amenities,nearby,negotiable,price,currency,published_date,description,date_scraped,url
0,4jYx9,olx,private,resale,Tashkent Region,Chilanzar District,1.0,20.0,6.0,29.0,...,0.0,"Internet, Refrigerator, TV, Air Conditioning, ...","Hospital, Clinic, Playground, Kindergarten, Bu...",1,60000.0,USD,01/05/2026,Описание Уй локацияси зур жойда жойлашган. Мир...,2026-05-02,https://www.olx.uz/d/obyavlenie/1-hona-metro-o...
1,4myoT,olx,private,NaN,Tashkent Region,Yunusabad District,2.0,80.0,NaN,80.0,...,1.0,"Internet, Refrigerator, TV, Air Conditioning, ...","Hospital, Clinic, Playground, Kindergarten, Bu...",0,7865000.0,UZS,30/04/2026,Описание Сдается в аренду 2/6/13 Юнусабад 13 к...,2026-05-02,https://www.olx.uz/d/obyavlenie/sdaetsya-v-are...
2,4lvjL,olx,business,new building,Tashkent Region,Yunusabad District,3.0,NaN,NaN,80.0,...,0.0,"Internet, Refrigerator, TV, Air Conditioning, ...","Hospital, Clinic, School, Playground, Kinderga...",0,220000.0,USD,29/04/2026,Описание Продается квартира !!! 3/6/9 80 кВ По...,2026-05-02,https://www.olx.uz/d/obyavlenie/srochno-prodae...
3,4aiOy,olx,private,NaN,Tashkent Region,Shaykhantakhur District,3.0,67.5,NaN,67.5,...,0.0,"Internet, Refrigerator, TV, Air Conditioning, ...",NaN,0,1400.0,USD,02/05/2026,"Описание Квартира в новостройке Tashkent City,...",2026-05-02,https://www.olx.uz/d/obyavlenie/sdaetsya-v-are...
4,4mAGS,olx,business,resale,Tashkent Region,Yashnabad District,3.0,50.0,NaN,78.0,...,1.0,"Internet, Telephone, Refrigerator, TV, Air Con...","Hospital, Clinic, Playground, Kindergarten, Bu...",0,120000.0,USD,02/05/2026,Описание Продается своя квартира 3/1/4 Кирпичн...,2026-05-02,https://www.olx.uz/d/obyavlenie/kvartira-3-1-4...


## A. Malumotna uchun shartnoma

# Ma'lumotlar lug'ati — OLX.uz kvartiralar dataseti
 
> **Hajmi:** 18 000+ qator × 28 ustun  
> **Manba:** OLX.uz — kvartiralar bo'limi (`/nedvizhimost/kvartiry/`)  
> **Skraping sanasi:** 2026-04-15 - hozir

## B. Ma'lumotlarni qabul qilish va dastlabki tekshiruvlar

### B.1. Asosiy qabul qilish ro'yxati:

In [8]:
print(f'File nomi: olx_apartments.csv')
print(f'Qatorlar soni: {df.shape[0]}')
print(f'Ustunlar soni: {df.shape[1]}')
print(f"Ustun nomlari malumot ustuniga o'xshashmimi: (Ha/yoq): Ha")

File nomi: olx_apartments.csv
Qatorlar soni: 67219
Ustunlar soni: 29
Ustun nomlari malumot ustuniga o'xshashmimi: (Ha/yoq): Ha


### B.2. Noyoblik + dublikatlar (Dunyo talabi)

In [9]:
before = len(df)
after = df.duplicated().sum()
print(f"Dublikatlar soni: {after}")
print(f'Noyoblik soni: {before-after}')
df_cleaned = df.drop_duplicates()

Dublikatlar soni: 3269
Noyoblik soni: 63950


### B.3. Natija yaxlitligini tekshirish
  
>Faqatgina ko'zlangan malumonni o'z ichaga oladimi? (yani narxni)  
* 2 hil turdaki valyuta aralashgan UZB va USD.
>Biror yoqolgan qiymatlar bormi?
* Bor 50 rotiq
>G‘alati yozilgan (imlosi noto‘g‘ri) so‘zlar bormi?
* Hozircha kuzatilmadi

## C. Malumot sifati (Nima hato ekanligini isbotlash)
  
### C.1. Yetishmayotgan Qiymatlar Profili
 
**Dataset hajmi:** 18,267 qator (`price` ustunidagi 40 ta bo'sh qiymat o'chirilgandan keyin)
 
---
 
## Yetishmayotgan Qiymatlar Jadvali
 
| Ustun | Yetishmaydi (n) | % | Qabul qilinadimi? | Nega yetishmayotgan bo'lishi mumkin? |
|---|---|---|---|---|
| listing_id | 0 | 0.0% | ✅ Ha | — |
| seller_type | 1 | 0.0% | ✅ Ha | Kichik ma'lumot kiritish xatosi |
| housing_type | 5,956 | 32.6% | ⚠️ Balki | Forma to'ldirishda majburiy emas |
| region | 1 | 0.0% | ✅ Ha | Kichik ma'lumot kiritish xatosi |
| district | 1 | 0.0% | ✅ Ha | Kichik ma'lumot kiritish xatosi |
| rooms | 1 | 0.0% | ✅ Ha | Kichik ma'lumot kiritish xatosi |
| living_area_m2 | 12,862 | 70.4% | ❌ Yo'q | Sotuvchilar ko'pincha maydon ma'lumotini qoldirmaydi |
| kitchen_area_m2 | 14,775 | 80.9% | ❌ Yo'q | Alohida ko'rsatilmaydi; ixtiyoriy maydon |
| total_area_m2 | 4 | 0.0% | ✅ Ha | Deyarli to'liq |
| floor | 2 | 0.0% | ✅ Ha | Deyarli to'liq |
| total_floors | 1 | 0.0% | ✅ Ha | Deyarli to'liq |
| building_type | 4,488 | 24.6% | ⚠️ Balki | Sotuvchi har doim bilmasligi mumkin |
| layout | 6,378 | 34.9% | ⚠️ Balki | Ixtiyoriy maydon; subyektiv tasnif |
| build_year | 13,785 | 75.5% | ❌ Yo'q | Eski binolarda hujjat yo'q bo'lishi mumkin |
| ceiling_height | 12,486 | 68.4% | ❌ Yo'q | E'lonlarda kamdan-kam ko'rsatiladi |
| bathroom | 5,057 | 27.7% | ⚠️ Balki | Ko'p formatlarda ixtiyoriy |
| furnished | 3 | 0.0% | ✅ Ha | Deyarli to'liq |
| renovation | 3,370 | 18.4% | ⚠️ Balki | Sotuvchi ta'mirlash holatini ko'rsatmasligi mumkin |
| commission | 1 | 0.0% | ✅ Ha | Deyarli to'liq |
| amenities | 7,045 | 38.6% | ⚠️ Balki | Ixtiyoriy maydon; har doim to'ldirilmaydi |
| nearby | 6,523 | 35.7% | ⚠️ Balki | Ixtiyoriy qo'shimcha maydon |
| negotiable | 0 | 0.0% | ✅ Ha | — |
| price | 40 | 0.2% | ❌ Yo'q | Maqsad ustun — to'liq bo'lishi shart; o'chirildi |
| currency | 0 | 0.0% | ✅ Ha | — |
| published_date | 1 | 0.0% | ✅ Ha | Kichik ma'lumot kiritish xatosi |
| description | 1 | 0.0% | ✅ Ha | Deyarli to'liq |
| date_scraped | 0 | 0.0% | ✅ Ha | — |
| url | 0 | 0.0% | ✅ Ha | — |
 
---
 
## Yetishmayotgan Qiymatlar Tahlili
 
### Yetishmayotgan qiymatlar tasodifiy yoki tizimliymi?
 
Bu datasetdagi yetishmayotgan qiymatlar **to'liq tasodifiy emas**. `living_area_m2` (70.4%), `kitchen_area_m2` (80.9%), `build_year` (75.5%) va `ceiling_height` (68.4%) ustunlarida qiymatlarning aksariyati yo'q. Bu **tizimli muammo**ni ko'rsatadi: ushbu maydonlar platforma formasida ixtiyoriy bo'lgan yoki sotuvchilar ularni to'ldirishni istamagan.
 
### Yetishmayotgan qiymatlar narx (price) bilan bog'liqmi?
 
Jismoniy xususiyatlari (maydon, qurilish yili, shift balandligi) ko'rsatilmagan e'lonlar **past sifatli yoki norasmiy** bo'lishi mumkin, bu esa narxlarning ham past bo'lishiga olib keladi. Agar shunday bo'lsa, bu ustunlarni o'rtacha qiymat bilan to'ldirish (imputation) **narx bashoratlarini yuqori tomonga siljitishi** mumkin.
 
### Guruhlar bo'yicha yetishmayotgan % ni tekshirish (tavsiya)
 
```python
# housing_type bo'yicha yetishmayotgan %
df.groupby('housing_type').apply(lambda x: x.isnull().mean() * 100)
 
# region bo'yicha yetishmayotgan %
df.groupby('region').apply(lambda x: x.isnull().mean() * 100)
```
 
### Xulosa
 
Yetishmayotgan qiymatlar **qisman tizimli** xarakterga ega: `living_area_m2`, `kitchen_area_m2`, `build_year` va `ceiling_height` ustunlarida 68–81% qiymat yo'q, chunki platforma ularni ixtiyoriy maydon sifatida ko'rsatadi. Bu oddiy xato emas — bu sotuvchilarning e'lon to'ldirishdagi tarkibiy kamchiligidir. Modellashtirish oldidan ushbu ustunlar uchun qaror qabul qilish kerak: **ustunlarni o'chirish**, **ehtiyotkorlik bilan to'ldirish (imputation)**, yoki **yetishmayotganlikni alohida belgi (binary flag) sifatida saqlash**.

### Amal qiladigan oralag'i columnlar uchun
  
---
 
| Ustun | Ma'nosi | Turi | Mumkin qiymatlar / diapazon | O'lchov birligi | Izohlar |
| --- | --- | --- | --- | --- | --- |
| `listing_id` | E'lon unikal identifikatori | string | `'4jYx9'` kabi 5 belgili alfanumerik kod | — | OLX tomonidan avtomatik beriladi; asosiy kalit (primary key) sifatida ishlatiladi |
| `seller_type` | Sotuvchi turi | kategoriya | `private`, `business` | — | `private` — uy egasi o'zi; `business` — agentlik yoki firma |
| `housing_type` | Uy-joy turi | kategoriya | `resale`, `new building` | — | 2 834 ta qiymat yo'q; ko'pchilik `resale` |
| `region` | Viloyat | kategoriya | `Tashkent Region` va boshqa viloyatlar | — | Hozirgi ma'lumotlarda asosan Toshkent viloyati |
| `district` | Tuman / mahalla | kategoriya | `Chilanzar District`, `Yunusabad District` va boshqalar | — | Narx prognozi uchun eng muhim ustunlardan biri |
| `rooms` | Xonalar soni | raqam | `1` — `6+` | dona | Butun son kutiladi; modelda kategoriya sifatida ham ishlatish mumkin |
| `living_area_m2` | Yashash maydoni | raqam | taxminan `10` — `200` | m² | 5 777 ta qiymat yo'q (66%); mediana bilan to'ldirish + `missing` flagi tavsiya etiladi |
| `kitchen_area_m2` | Oshxona maydoni | raqam | taxminan `4` — `30` | m² | 6 656 ta qiymat yo'q (76%); yuqori yo'qotish darajasi |
| `total_area_m2` | Umumiy maydon | raqam | taxminan `15` — `500` | m² | Faqat 1 ta qiymat yo'q; narx prognozi uchun asosiy ustun |
| `floor` | Qavat | raqam | `1` — `30+` | qavat | 1 ta qiymat yo'q; mediana bilan to'ldirish yetarli |
| `total_floors` | Binodagi jami qavatlar soni | raqam | `2` — `30+` | qavat | Qiymat yo'q emas; `floor / total_floors` nisbati yaratish tavsiya etiladi |
| `building_type` | Bino turi | kategoriya | `panel`, `brick`, `monolith` | — | 2 029 ta qiymat yo'q; eng ko'p uchraydigan qiymat bilan to'ldirish |
| `layout` | Xonalar tartibi | kategoriya | `adjacent`, `separate`, `Смежно-раздельная` | — | Aralash til (rus/ingliz) bor — tozalash kerak; 2 840 ta qiymat yo'q |
| `build_year` | Qurilgan yil | raqam | `1950` — `2026` | yil | 6 197 ta qiymat yo'q; `building_age = 2026 - build_year` ustuni hosil qilish tavsiya etiladi |
| `ceiling_height` | Shift balandligi | raqam | `2.5` — `4.0` | metr | 5 683 ta qiymat yo'q; mediana bilan to'ldirish |
| `bathroom` | Hammom turi | kategoriya | `separate`, `combined` | — | 2 312 ta qiymat yo'q; eng ko'p uchraydigan qiymat bilan to'ldirish |
| `furnished` | Mebel mavjudligi | ikkilik (binary) | `1` — mebel bor, `0` — yo'q | — | Faqat 2 ta qiymat yo'q |
| `renovation` | Ta'mirlash holati | kategoriya | `euro renovation`, `average condition`, `designer renovation` | — | 1 538 ta qiymat yo'q; narxga sezilarli ta'sir ko'rsatadi |
| `commission` | Vositachi komissiyasi | ikkilik (binary) | `0` — yo'q, `1` — bor | — | Qiymat yo'q emas; `1` bo'lsa agentlik orqali sotiladi |
| `amenities` | Jihozlar ro'yxati | vergul bilan ajratilgan string | `Internet`, `TV`, `Air Conditioning`, `Balcony`, `Washing Machine` va boshqalar | — | 3 187 ta qiymat yo'q = jihozlar ko'rsatilmagan; multi-hot encoding tavsiya etiladi |
| `nearby` | Yaqin atrofdagi ob'ektlar | vergul bilan ajratilgan string | `School`, `Bus Stop`, `Park`, `Hospital`, `Supermarket` va boshqalar | — | 2 874 ta qiymat yo'q; multi-hot encoding tavsiya etiladi |
| `negotiable` | Narx kelishiladi | ikkilik (binary) | `1` — ha, `0` — yo'q | — | Qiymat yo'q emas |
| `price` | Narx | raqam | `500` — `2 000 000` (USDga o'tkazilgandan keyin) | USD yoki UZS | 13 ta qiymat yo'q — bu qatorlarni o'chirish kerak; avval valyutani normallashtirish shart |
| `currency` | Valyuta | kategoriya | `USD`, `UZS` | — | Modelga kirishdan oldin hammasi USDga aylantiriladi (1 USD ≈ 12 700 UZS) |
| `published_date` | E'lon joylashtirilgan sana | sana | `DD/MM/YYYY` formatida | sana | Modelda `days_since_posted` ustuniga aylantirish mumkin |
| `description` | E'lon matni | erkin matn | Rus va o'zbek tillarida | — | Modelga kiritilmaydi; NLP tahlili uchun alohida ishlatish mumkin |
| `date_scraped` | Skraping sanasi | sana | `YYYY-MM-DD` formatida | sana | Barcha qatorlarda `2026-05-02`; modelda ishlatilmaydi |
| `url` | E'lon havolasi | string | `https://www.olx.uz/d/obyavlenie/...` | — | Modelda ishlatilmaydi; manba sifatida saqlab qo'yish tavsiya etiladi |
 
---
 
## Tozalash bo'yicha qisqacha yo'riqnoma
 
| Muammo | Ta'sirlangan ustunlar | Yechim |
| --- | --- | --- |
| Maqsadli qiymat yo'q | `price` (13 ta) | Qatorlarni o'chirish |
| Yuqori yo'qotish (>50%) | `living_area_m2`, `kitchen_area_m2`, `build_year`, `ceiling_height` | Mediana bilan to'ldirish + `_missing` flagi |
| O'rtacha yo'qotish | `housing_type`, `building_type`, `layout`, `bathroom`, `renovation` | Eng ko'p uchraydigan qiymat (mode) bilan to'ldirish |
| Aralash valyuta | `price`, `currency` | Hammasi USDga o'tkazish |
| Aralash til | `layout` | `Смежно-раздельная` → `mixed` deb standartlashtirish |
| Vergul bilan ajratilgan qiymatlar | `amenities`, `nearby` | Multi-hot encoding (har bir teg — alohida ustun) |

## D. Tozalash strategiyasi (nafaqat qadamlar-qarorlar)
  
### D.1 Standartlashtirish qoidalari (kategoriyaviy)
`seller_type` --> Faqatgina 2 dona categoria aniqlandi. Ortiqcha tekshiruv talab qilmaydi.  
`housing_type` --> 6 ga yaqin categoria aniqladi. Uchta rus tilodagi miqdorlar("новостройка", "Новостройка", "Новостройка.") --> new building  
`region` -- > Hammasi joyida.


In [10]:
df_cleaned['district'].value_counts()

district
Mirzo-Ulugbek District    10335
Yunusabad District         7365
Yakkasaray District        6648
Mirabad District           6203
Yashnabad District         5979
                          ...  
Касан                         1
Багдад                        1
Риштан                        1
Булакбаши                     1
Учкурган                      1
Name: count, Length: 180, dtype: int64

---

## Price malumot tozalash

In [11]:
df_cleaned[df_cleaned['price'].isnull()]

,listing_id,source,seller_type,housing_type,region,district,rooms,living_area_m2,kitchen_area_m2,total_area_m2,...,commission,amenities,nearby,negotiable,price,currency,published_date,description,date_scraped,url
1314,4mwtN,olx,private,NaN,Tashkent Region,Бектемирский район,3.0,45.0,12.0,66.0,...,0.0,"Internet, Refrigerator, TV, Cable TV, Washing ...","Hospital, Clinic, Playground, Kindergarten, Bu...",0,NaN,USD,29/04/2026,Описание Musaffo maskan 3/6/10 400$ ichida ha...,2026-05-02,https://www.olx.uz/d/obyavlenie/musaffo-maskan...
2126,4ms7L,olx,business,NaN,Tashkent Region,Mirzo-Ulugbek District,2.0,56.0,NaN,56.0,...,1.0,NaN,NaN,1,NaN,USD,01/05/2026,Описание Мирзо-Улугбекский район ЖК Akay City ...,2026-05-02,https://www.olx.uz/d/obyavlenie/akay-city-2-ko...
3926,4mCMT,olx,business,new building,Tashkent Region,Yashnabad District,2.0,NaN,NaN,60.0,...,1.0,NaN,NaN,0,NaN,USD,02/05/2026,Описание Срочно Продаётся Новостройка 2 комнат...,2026-05-02,https://www.olx.uz/d/obyavlenie/srochno-novost...
4431,4mCEI,olx,business,NaN,Tashkent Region,Yashnabad District,2.0,NaN,NaN,52.0,...,1.0,"Internet, Telephone, Refrigerator, TV, Air Con...","Hospital, Clinic, Playground, Kindergarten, Bu...",1,NaN,USD,02/05/2026,Описание Район: Яшнабод Ориентир: Better best ...,2026-05-02,https://www.olx.uz/d/obyavlenie/arenda-kvartir...
4624,4lDkg,olx,business,NaN,Обмен - Ферганская область,Обмен - Маргилан,3.0,56.0,15.0,56.0,...,0.0,"Telephone, Cable TV, TV, Internet","Entertainment, Bus Stop, School, Supermarket, ...",0,NaN,USD,02/05/2026,Описание Ijaraga beriladi yoki hovli uyga bart...,2026-05-02,https://www.olx.uz/d/obyavlenie/dom-ijaraga-be...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64445,3MdsV,olx,business,NaN,Обмен - Ташкент,Обмен - Алмазарский район,3.0,NaN,NaN,50.0,...,0.0,NaN,"Playground, School, Entertainment, Hospital, C...",0,NaN,USD,16/05/2026,Описание Меняю свои 1-комн + 2-комн квартиры с...,2026-05-16,https://www.olx.uz/d/obyavlenie/menyayu-2-1-4-...
64669,4n1pO,olx,business,NaN,Tashkent Region,Mirzo-Ulugbek District,1.0,42.0,6.0,42.0,...,0.0,"Cable TV, TV, Telephone, Balcony, Washing Mach...","Entertainment, Bus Stop, School, Supermarket, ...",0,NaN,USD,16/05/2026,Описание Звоните по телефону более подробно об...,2026-05-16,https://www.olx.uz/d/obyavlenie/srochno-arenda...
65109,44Wmm,olx,private,new building,Tashkent Region,Назарбек,2.0,72.0,15.0,72.0,...,0.0,"Cable TV, Internet, Kitchen","Kindergarten, Restaurant, Cafe, Playground, Sc...",1,NaN,USD,16/05/2026,Описание Zamona toyxonasi Qizgaldoq magalla ya...,2026-05-16,https://www.olx.uz/d/obyavlenie/zamona-qizgald...
65644,4eZQL,olx,business,resale,Tashkent Region,Алмазарский район,4.0,NaN,NaN,84.0,...,0.0,"Air Conditioning, Refrigerator, Washing Machin...","Restaurant, Cafe, Kindergarten, Parking, Bus S...",1,NaN,USD,16/05/2026,Описание Uy sotiladi Olmazor Tumani Qoraqamish...,2026-05-16,https://www.olx.uz/d/obyavlenie/uy-sotiladi-97...


In [12]:
df_cleaned[df_cleaned['currency'] == "UZS"]

,listing_id,source,seller_type,housing_type,region,district,rooms,living_area_m2,kitchen_area_m2,total_area_m2,...,commission,amenities,nearby,negotiable,price,currency,published_date,description,date_scraped,url
1,4myoT,olx,private,NaN,Tashkent Region,Yunusabad District,2.0,80.0,NaN,80.00,...,1.0,"Internet, Refrigerator, TV, Air Conditioning, ...","Hospital, Clinic, Playground, Kindergarten, Bu...",0,7865000.0,UZS,30/04/2026,Описание Сдается в аренду 2/6/13 Юнусабад 13 к...,2026-05-02,https://www.olx.uz/d/obyavlenie/sdaetsya-v-are...
24,49bVu,olx,private,NaN,Samarkand Region,Лаиш,2.0,65.0,NaN,10.00,...,1.0,"Internet, Telephone, Refrigerator, TV, Air Con...","Hospital, Clinic, School, Playground, Kinderga...",0,4000000.0,UZS,02/05/2026,Описание Samarqand shahar Qorasuv massiv 2 xon...,2026-05-02,https://www.olx.uz/d/obyavlenie/bez-agestvo-ij...
29,4lwkw,olx,private,NaN,Khorezm Region,Ургенч,3.0,49.0,10.00,62.00,...,0.0,"Refrigerator, TV, Air Conditioning, Washing Ma...","Hospital, Clinic, School, Playground, Kinderga...",0,2800000.0,UZS,02/05/2026,Описание Ургенч шахар марказида мед институтга...,2026-05-02,https://www.olx.uz/d/obyavlenie/kvartira-izhar...
30,3nWuH,olx,private,resale,Samarkand Region,Samarkand,5.0,86.0,12.00,100.00,...,0.0,"Kitchen, Balcony","Hospital, Clinic, Playground, Kindergarten, Bu...",0,900000000.0,UZS,02/05/2026,Описание 5 хона 6 хона килинган квартира сотил...,2026-05-02,https://www.olx.uz/d/obyavlenie/6-honalik-kvar...
33,3It0w,olx,private,NaN,Tashkent Region,Yashnabad District,4.0,100.0,12.00,92.00,...,1.0,"Internet, Telephone, Refrigerator, TV, Air Con...","School, Playground, Kindergarten, Bus Stop, Pa...",1,1300000.0,UZS,02/05/2026,"Описание Квартирага боллани куяман, 1.300 000...",2026-05-02,https://www.olx.uz/d/obyavlenie/kvartiraga-ish...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67143,3PxcX,olx,private,NaN,Tashkent Region,Chilanzar District,2.0,50.0,12.00,50.00,...,0.0,"Refrigerator, Kitchen, Balcony","Hospital, Clinic, Playground, Kindergarten, Bu...",0,600000.0,UZS,19/05/2026,Описание Chilonzor 19kv da 2xonali kvartiraga ...,2026-05-19,https://www.olx.uz/d/obyavlenie/chilonzorda-st...
67146,4bGd0,olx,private,NaN,Jizzakh Region,Галлаарал,2.0,50.0,10.00,50.00,...,0.0,"Refrigerator, Kitchen, Balcony","Hospital, Clinic, Playground, Kindergarten, Bu...",0,600000.0,UZS,19/05/2026,Описание Qizlar bn sheriklikka oʻqiydigan ishl...,2026-05-19,https://www.olx.uz/d/obyavlenie/chilonzor-19-k...
67176,3IuTp,olx,business,NaN,Tashkent Region,Yunusabad District,3.0,70.0,10.00,84.00,...,0.0,"Telephone, Washing Machine, Internet, Cable TV...","School, Supermarket, Shops, Park, Green Area, ...",1,800000.0,UZS,18/05/2026,Описание Assalomu alaykum kvarteraga ugil bola...,2026-05-19,https://www.olx.uz/d/obyavlenie/ogil-bollarga-...
67196,4fdGN,olx,private,resale,Navoiy Region,Navoiy,2.0,30.2,10.77,54.36,...,0.0,"Kitchen, Balcony","Hospital, Clinic, School, Playground, Kinderga...",1,400000000.0,UZS,19/05/2026,Описание Квартира Спутник қўрғонида жойлашган ...,2026-05-19,https://www.olx.uz/d/obyavlenie/kvartira-sroch...


In [13]:
# Birinchi bo'lib narxi yo'q qatorlarni olib tashlaymiz
before1 = len(df_cleaned['price'])
df_cleaned.dropna(subset=['price'], inplace=True)
print(f"Qatorlar soni(Dublikat bilan): {before1}")
print(f"Olib tashlangan qator soni: {before1 - len(df_cleaned['price'])}")
print(f"Noyoblik soni: {len(df_cleaned['price'])}")

# Endi bir xil valyutaga o'tqazib olamiz
exchange_rate = 12700
df_cleaned['price_usd'] = df_cleaned['price']
df_cleaned.loc[df_cleaned['currency']=="UZS", 'price_usd'] = (
    df_cleaned.loc[df_cleaned['currency']=="UZS", 'price_usd'] / exchange_rate
).round(1)

Qatorlar soni(Dublikat bilan): 63950
Olib tashlangan qator soni: 166
Noyoblik soni: 63784


/var/folders/pn/2q111n451j10mgh135bwncf00000gn/T/ipykernel_11167/3476733231.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned.dropna(subset=['price'], inplace=True)
/var/folders/pn/2q111n451j10mgh135bwncf00000gn/T/ipykernel_11167/3476733231.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['price_usd'] = df_cleaned['price']


Hozirda 3 savdo turdaki uylar OLXda joylngan.  
Bu muammo OLX tomonidan amalga oshirilgan bo'lishi mumkin va natijada bu malumotni tozalash qiyinchiligiga olip keladi.  
Bu muammoni yechish uch ikki turdaki yechim mavjud. Biri narx orqali savdo turini aniqlash va ikkinchi esa commetaria orqali.

---

## `listing_id` malumot tozalash

In [14]:
print(f"listing_id dublikatlar soni: {len(df_cleaned)-len(df_cleaned['listing_id'].unique())}")

listing_id dublikatlar soni: 262


Qaysidir sabab orqali `listing_id` o'z ichiga dublikatlarni oladi.

In [15]:
print(f"Dublikatlar soni: {len(df_cleaned)}")
df_cleaned = df_cleaned.drop_duplicates(subset=['listing_id'])
print(f'Noyoblik soni dublikatlarni olib tashlangandam keyin: {len(df_cleaned)}')

Dublikatlar soni: 63784
Noyoblik soni dublikatlarni olib tashlangandam keyin: 63522


In [16]:
df_cleaned[df_cleaned.duplicated(subset='listing_id', keep=False)].sort_values(by = 'listing_id')

,listing_id,source,seller_type,housing_type,region,district,rooms,living_area_m2,kitchen_area_m2,total_area_m2,...,amenities,nearby,negotiable,price,currency,published_date,description,date_scraped,url,price_usd


---

## `seller_type` tozalash jarayoni

In [17]:
df_cleaned['seller_type'].isna().sum()

np.int64(0)

seller_type column toza va hech qanday xatolar mavjud emas


---

## `housing_type`  malumot tozalash

In [18]:
df_cleaned['housing_type'].value_counts()

housing_type
new building               23147
resale                     19132
Новостройка                    5
новостройка                    3
Новостройка.                   1
Кирпич                         1
Вторичка,кирпичный дом.        1
От застройщика                 1
Name: count, dtype: int64

In [19]:
print(f"missing - {round(df_cleaned['housing_type'].isna().sum()/len(df_cleaned)*100, 2)}%")

missing - 33.42%


In [20]:
valid_housing_type = {
    'new building': 'new building',
    'resale': 'resale',
    'Новостройка': 'new building',
    'новостройка': 'new building',
    'Новостройка.': 'new building',
    'Вторичка,кирпичный дом.': 'new building'
}

def validate_housing_type(text):
    if pd.isna(text):
        return np.nan

    if text in valid_housing_type:
        return valid_housing_type[text]
    else:
        return np.nan

    return text

df_cleaned['housing_type'] = df_cleaned['housing_type'].apply(validate_housing_type)

In [21]:
df_cleaned['housing_type'].value_counts()

housing_type
new building    23157
resale          19132
Name: count, dtype: int64

---

## `region` malumot tozalash

In [22]:
df_cleaned['region'].value_counts()

region
Tashkent Region               55809
Samarkand Region               2734
Bukhara Region                 2498
Fergana Region                  645
Navoiy Region                   608
Khorezm Region                  295
Kashkadarya Region              273
Republic of Karakalpakstan      160
Surxondaryo Region              137
Andijan Region                  123
Jizzakh Region                  101
Sirdaryo Region                  81
Namangan Region                  58
Name: count, dtype: int64

In [23]:
df_cleaned['region'].isna().sum()

np.int64(0)

Region column toza. 

----


## `district` tozalash jarayoni

In [24]:
df_cleaned['district'].value_counts()

district
Mirzo-Ulugbek District    10307
Yunusabad District         7317
Yakkasaray District        6621
Mirabad District           6181
Yashnabad District         5926
                          ...  
Джаркурган                    1
Канлыкуль                     1
Чимбай                        1
Риштан                        1
Учкурган                      1
Name: count, Length: 167, dtype: int64

----


## `rooms` tozalash jarayoni

In [25]:
df_cleaned['rooms'].value_counts()

rooms
2.0      26464
3.0      19698
1.0       9013
4.0       6293
5.0       1463
6.0        285
7.0        108
8.0         70
9.0         28
12.0        13
10.0        13
50.0         5
11.0         5
60.0         4
51.0         3
22.0         3
100.0        3
400.0        3
25.0         3
52.0         2
500.0        2
70.0         2
47.0         2
30.0         2
26.0         2
64.0         2
14.0         2
90.0         1
78.0         1
86.0         1
20.0         1
72.0         1
24.0         1
94.0         1
65.0         1
35.0         1
215.0        1
66.0         1
45.0         1
450.0        1
16.0         1
56.0         1
133.0        1
15.0         1
19.0         1
123.0        1
350.0        1
49.0         1
67.0         1
53.0         1
165.0        1
125.0        1
37.0         1
13.0         1
23.0         1
150.0        1
Name: count, dtype: int64

In [26]:
df_cleaned_1 = df_cleaned[df_cleaned['rooms'] < 7].copy()

In [27]:
df_cleaned_1['rooms'].value_counts()

rooms
2.0    26464
3.0    19698
1.0     9013
4.0     6293
5.0     1463
6.0      285
Name: count, dtype: int64

Man 6 xonadan katta bo'lgan barcha elonlarni olib tashlashga qaror qildim.  
Kuzatuvlarga asoslanib shuni ayta olamanki. 6+ xonadonlar soni 500 ga yaqin va ularning ko'pchiligi xaqiqatga yaqin emas.

----

## `living_area_m2` tozalash jarayoni

In [35]:
print(f"missing: {round((df_cleaned_1['living_area_m2'].isna().sum()/len(df_cleaned_1['living_area_m2']))*100)}")

missing: 68


Raughly 68% of data is missing.

In [36]:
df_cleaned_1['living_area_m2'].describe()

count    20262.000000
mean        62.256371
std         46.481564
min          1.000000
25%         40.000000
50%         56.000000
75%         73.000000
max        999.000000
Name: living_area_m2, dtype: float64

Natijaga asoslanib shuni ayta olamizki `living_area_m2` judaham ishlatishga yaroqsiz.  
OLX `living_area_m2` ni optional qilib qoygan shu sabali `68 foiz` data bo'sh qoldirilgan.  
Judaham ko'b sotuvchilar taxminan to'ldirgan bo'lishlari mumkin.  
Katta extimollik bilan `living_area_m2` analysisda va predictive model uchun yaroqsis.

In [40]:
df_cleaned_1 = df_cleaned_1.drop(columns=['living_area_m2'])

## `kitchen_area_m2` Tozalash jarayoni

In [37]:
print(f"missing: {round((df_cleaned_1['kitchen_area_m2'].isna().sum()/len(df_cleaned_1['kitchen_area_m2']))*100)}")

missing: 78


In [38]:
df_cleaned_1['kitchen_area_m2'].describe()

count    14201.000000
mean        16.883266
std         41.723017
min          0.300000
25%          7.880000
50%         11.000000
75%         15.000000
max        999.000000
Name: kitchen_area_m2, dtype: float64

78% data to'ldirilmagan.  
Natijaga asoslanganda shu aniqki kiritilga raqamlar xaqiqatga yaqin emas.  


In [42]:
df_cleaned_1 = df_cleaned_1.drop(columns="kitchen_area_m2")

`total_area_m2` Tozalash jarayoni